
# Task 3 – Model Improvement

## SWYNEX Technologies – Machine Learning Internship

**Project:** Titanic Survival Prediction

**Objective:** Improve the performance of the Random Forest machine learning model using feature engineering, hyperparameter tuning and cross-validation.

**Previous Model:** Random Forest Classifier

**Previous Accuracy:** 82.12%

**Dataset:** Titanic Dataset


In [19]:
# Import required libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    cross_val_score
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("Libraries imported successfully!")

Libraries imported successfully!


In [20]:
# Load Titanic dataset

df = pd.read_csv(
    "https://raw.githubusercontent.com/hari-vasu-186/Swynex-Intern-/main/Task-3-Model-Improvement/data/Titanic-Dataset.csv"
)

print("Dataset loaded successfully!")

display(df.head())

Dataset loaded successfully!


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [21]:
# Explore dataset

print("Dataset Shape:", df.shape)

print("\nDataset Information:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum())

Dataset Shape: (891, 12)

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB

Missing Values:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare     

In [22]:
# Feature Engineering

# Create FamilySize feature
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

# Create IsAlone feature
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

# Extract passenger title from Name
df['Title'] = df['Name'].str.extract(
    r',\s*([^\.]+)\.',
    expand=False
)

print("Feature engineering completed!")

display(df[['Name', 'FamilySize', 'IsAlone', 'Title']].head())

Feature engineering completed!


,Name,FamilySize,IsAlone,Title
0,"Braund, Mr. Owen Harris",2,0,Mr
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",2,0,Mrs
2,"Heikkinen, Miss. Laina",1,1,Miss
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",2,0,Mrs
4,"Allen, Mr. William Henry",1,1,Mr


In [23]:
# Data Preprocessing

# Fill missing numerical values
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Fare'] = df['Fare'].fillna(df['Fare'].median())

# Fill missing categorical values
df['Embarked'] = df['Embarked'].fillna(
    df['Embarked'].mode()[0]
)

# Fill missing Cabin with Unknown
df['Cabin'] = df['Cabin'].fillna('Unknown')

# Remove unnecessary columns
df = df.drop(
    columns=['PassengerId', 'Name', 'Ticket', 'Cabin']
)

# Group less common titles
df['Title'] = df['Title'].replace(
    ['Lady', 'Countess', 'Capt', 'Col', 'Don',
     'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer',
     'Dona'],
    'Other'
)

df['Title'] = df['Title'].replace(
    ['Mlle', 'Ms'],
    'Miss'
)

df['Title'] = df['Title'].replace(
    'Mme',
    'Mrs'
)

print("Data preprocessing completed!")

print("\nMissing values:")
print(df.isnull().sum())

display(df.head())

Data preprocessing completed!

Missing values:
Survived      0
Pclass        0
Sex           0
Age           0
SibSp         0
Parch         0
Fare          0
Embarked      0
FamilySize    0
IsAlone       0
Title         0
dtype: int64


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,FamilySize,IsAlone,Title
0,0,3,male,22.0,1,0,7.2500,S,2,0,Mr
1,1,1,female,38.0,1,0,71.2833,C,2,0,Mrs
2,1,3,female,26.0,0,0,7.9250,S,1,1,Miss
3,1,1,female,35.0,1,0,53.1000,S,2,0,Mrs
4,0,3,male,35.0,0,0,8.0500,S,1,1,Mr


In [24]:
# Encode categorical variables

df['Sex'] = df['Sex'].map({
    'male': 0,
    'female': 1
})

df = pd.get_dummies(
    df,
    columns=['Embarked', 'Title'],
    dtype=int
)

print("Categorical encoding completed!")

display(df.head())

print("\nFinal dataset shape:", df.shape)

Categorical encoding completed!


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,FamilySize,IsAlone,Embarked_C,Embarked_Q,Embarked_S,Title_Master,Title_Miss,Title_Mr,Title_Mrs,Title_Other,Title_the Countess
0,0,3,0,22.0,1,0,7.2500,2,0,0,0,1,0,0,1,0,0,0
1,1,1,1,38.0,1,0,71.2833,2,0,1,0,0,0,0,0,1,0,0
2,1,3,1,26.0,0,0,7.9250,1,1,0,0,1,0,1,0,0,0,0
3,1,1,1,35.0,1,0,53.1000,2,0,0,0,1,0,0,0,1,0,0
4,0,3,0,35.0,0,0,8.0500,1,1,0,0,1,0,0,1,0,0,0



Final dataset shape: (891, 18)


In [25]:
# Separate features and target

X = df.drop('Survived', axis=1)
y = df['Survived']

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (891, 17)
Target shape: (891,)


In [26]:
# Split dataset into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

Training features: (712, 17)
Testing features: (179, 17)
Training target: (712,)
Testing target: (179,)


In [27]:
# Train Baseline Random Forest Model

baseline_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

baseline_model.fit(X_train, y_train)

# Make predictions
baseline_predictions = baseline_model.predict(X_test)

print("Baseline Random Forest trained successfully!")

Baseline Random Forest trained successfully!


In [28]:
# Evaluate Baseline Random Forest

baseline_accuracy = accuracy_score(
    y_test, baseline_predictions
)

baseline_precision = precision_score(
    y_test, baseline_predictions
)

baseline_recall = recall_score(
    y_test, baseline_predictions
)

baseline_f1 = f1_score(
    y_test, baseline_predictions
)

print("BASELINE RANDOM FOREST RESULTS")
print("--------------------------------")
print("Accuracy:", round(baseline_accuracy, 4))
print("Precision:", round(baseline_precision, 4))
print("Recall:", round(baseline_recall, 4))
print("F1 Score:", round(baseline_f1, 4))

BASELINE RANDOM FOREST RESULTS
--------------------------------
Accuracy: 0.8212
Precision: 0.7761
Recall: 0.7536
F1 Score: 0.7647


In [ ]:
# Hyperparameter Tuning using GridSearchCV

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10, 15],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest Cross-Validation Accuracy:")
print(round(grid_search.best_score_, 4))

Fitting 5 folds for each of 108 candidates, totalling 540 fits


In [ ]:
# Train the improved Random Forest model

best_model = grid_search.best_estimator_

improved_predictions = best_model.predict(X_test)

print("Improved Random Forest model trained successfully!")

In [ ]:
# Evaluate the improved model

improved_accuracy = accuracy_score(y_test, improved_predictions)
improved_precision = precision_score(y_test, improved_predictions)
improved_recall = recall_score(y_test, improved_predictions)
improved_f1 = f1_score(y_test, improved_predictions)

print("Improved Random Forest Model Performance")
print("----------------------------------------")
print("Accuracy :", round(improved_accuracy, 4))
print("Precision:", round(improved_precision, 4))
print("Recall   :", round(improved_recall, 4))
print("F1 Score :", round(improved_f1, 4))

In [ ]:
# Compare model performance

comparison = pd.DataFrame({
    'Model': [
        'Task 2 Random Forest',
        'Task 3 Baseline Random Forest',
        'Task 3 Improved Random Forest'
    ],
    'Accuracy': [
        0.8212,
        baseline_accuracy,
        improved_accuracy
    ],
    'Precision': [
        0.8136,
        baseline_precision,
        improved_precision
    ],
    'Recall': [
        0.6957,
        baseline_recall,
        improved_recall
    ],
    'F1 Score': [
        0.7500,
        baseline_f1,
        improved_f1
    ]
})

comparison.round(4)

In [ ]:
# Feature Importance

feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': best_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by='Importance', ascending=False
)

plt.figure(figsize=(10, 6))
sns.barplot(
    x='Importance',
    y='Feature',
    data=feature_importance
)

plt.title('Feature Importance - Improved Random Forest')
plt.xlabel('Importance')
plt.ylabel('Features')
plt.show()

In [ ]:
# Confusion Matrix

cm = confusion_matrix(y_test, improved_predictions)

plt.figure(figsize=(6, 4))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Not Survived', 'Survived'],
    yticklabels=['Not Survived', 'Survived']
)

plt.title('Confusion Matrix - Improved Random Forest')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
# Classification Report

print("Classification Report - Improved Random Forest")
print("-----------------------------------------------")

print(classification_report(
    y_test,
    improved_predictions,
    target_names=['Not Survived', 'Survived']
))

## Conclusion

In this task, the Random Forest model was improved using feature engineering and hyperparameter tuning with GridSearchCV.

New features such as FamilySize, IsAlone, and Title were created to improve prediction performance. GridSearchCV was used to identify the best hyperparameters using 5-fold cross-validation.

The improved Random Forest model achieved:
- Accuracy: 82.68%
- Precision: 81.67%
- Recall: 71.01%
- F1-score: 75.97%
- Cross-validation accuracy: 83.01%

The results show a small improvement in test accuracy compared with the previous Task 2 Random Forest model. The model also achieved better precision, while the Task 3 baseline model achieved a higher F1-score.

Overall, feature engineering and hyperparameter tuning helped improve the Titanic survival prediction model.
